# Rotated Anisotropic Gaussian Derivative Bank

This filter builds the angular bank by rotating anisotropic first-derivative Gaussian kernels.

## How It Works

Each kernel differentiates in one direction and smooths with different widths along and across that direction. This is useful when you want directional edge responses from explicitly rotated kernels.

### Reference

The filter is closely related to first-order steerable filters from Freeman and Adelson, 1991. DOI [10.1109/34.93808](https://doi.org/10.1109/34.93808). Perona's deformable-kernel work is also related. DOI [10.1109/34.391394](https://doi.org/10.1109/34.391394).

## Setup

These notebooks use filters whose kernels are explicitly rotated through the angle grid. The output is already an orientation bank.


In [ ]:
# Imports.
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, get_filter_definition, run_orientation_bank

## Filter Settings

`sigma_parallel` controls the width along the derivative direction. `sigma_perpendicular` controls the width across it.

In [ ]:
# Choose the rotated-kernel bank.
angle_count = 12
sigma_parallel = 1.0
sigma_perpendicular = 8.0
definition = get_filter_definition(
    "anisotropic_gaussian_orientation_bank",
    angle_count=angle_count,
    sigma_parallel=sigma_parallel,
    sigma_perpendicular=sigma_perpendicular,
)
path = ExecutionPath.ORIENTATION_BANK

## Test Image

The default image is a 1024 by 1024 black-to-white step edge. The commented lines show the smallest path-based image import you can use instead.


In [ ]:
# Build the test image.
size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

# To use your own image instead, uncomment these lines.
# image_path = "path/to/image.png"
# image = torch.as_tensor(plt.imread(image_path), dtype=torch.float32)
# if image.ndim == 3:
#     image = image[..., :3].mean(dim=-1)
# image = image.unsqueeze(0)

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

## Run And Plot

The result contains one response image per angle. Change `angle_index` to inspect a different rotated kernel and response.


In [ ]:
# Time the orientation-bank call.
start = time.perf_counter()
result = run_orientation_bank(definition, image, path=path, boundary=definition.default_boundary)
elapsed = time.perf_counter() - start
print(f"{elapsed:.4f} seconds")

angle_index = 0
kernel = definition.require_implementation().orientation_kernels[angle_index]
response = result.responses[0, angle_index]
kernel_limit = float(kernel.abs().max())
response_limit = float(response.abs().max())

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(kernel, cmap="gray", vmin=-kernel_limit, vmax=kernel_limit)
axes[0].set_title("kernel")
axes[1].imshow(response, cmap="gray", vmin=-response_limit, vmax=response_limit)
axes[1].set_title(f"angle {float(result.angles[angle_index]):.2f}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()